# Exploratory Data Analysis: Steam's Highest-Revenue Games of 2024

**Holon Institute of Technology (HIT) — Faculty of Computer Science**

| | |
|---|---|
| **Course** | Introduction to Data Science |
| **Assignment** | 1 — Exploratory Data Analysis (EDA) |
| **Student name** | *[TO BE COMPLETED]* |
| **ID number** | *[TO BE COMPLETED]* |
| **Lecturer** | Dr. Ori Itai |
| **Teaching assistant** | Hanit Ohayon Hadad |
| **Submission** | Individual |
| **Due date** | 02.08.2026 |

---

## Table of Contents

1. [Introduction and Objective](#1-introduction-and-objective)
2. [Dataset Selection](#2-dataset-selection)
3. [Meta-Analysis of the Data](#3-meta-analysis-of-the-data)
4. [Data Quality and Completeness](#4-data-quality-and-completeness)
5. [Univariate Analysis](#5-univariate-analysis)
6. [Correlations and Relationships](#6-correlations-and-relationships)
7. [Index Analysis](#7-index-analysis)
8. [Insights and the Data Story](#8-insights-and-the-data-story)
9. [Extensions](#9-extensions)

## 1. Introduction and Objective

This notebook is an exploratory analysis of the 1,500 highest-revenue games
released on Steam during 2024.

The objective is not to produce charts, but to understand the dataset as a
**system**: its internal structure, the assumptions hidden inside it, and the
limitations and biases those assumptions impose on any conclusion drawn from
it. The analysis is guided throughout by a single principle:

> Data is not "truth". It is the output of a measurement process.

Each section therefore asks not only *what the numbers say*, but also **what was
measured, what is missing, what is biased, and what has been artificially
centred**. Where a finding contradicts what one would intuitively expect, the
contradiction is stated explicitly rather than smoothed over.

## 2. Dataset Selection

### 2.1 Verification of the Selection Requirements

The assignment defines four criteria a chosen dataset must satisfy: it must be
tabular, contain at least 1,000 rows and at least 10 columns, and combine
numeric, temporal and categorical variables.

These criteria are verified below rather than asserted, because the same checks
also produce the first structural facts about the data. The semantic role of
each column — numeric, temporal, categorical or identifier — is declared once in
`src/data_loading.py` and reused for the rest of the notebook, so that the
analysis never depends on a column list retyped in several places.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# This notebook lives in <project>/notebooks while the helper modules live in
# <project>/src. Adding the project directory to the import path lets the
# notebook import them regardless of the directory Jupyter was started from.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.data_loading import (
    NUMERIC_COLUMNS,
    RAW_DATA_PATH,
    build_column_inventory,
    load_raw_dataset,
    parse_release_date,
    summarise_entity_columns,
    verify_selection_requirements,
)
from src.cleaning import apply_quality_fixes, summarise_cleaning_effect
from src.univariate import (
    compare_outlier_methods,
    describe_numeric,
    expand_entity_column,
    summarise_categorical,
    values_needed_for_coverage,
)
from src.quality_checks import (
    build_cardinality_summary,
    compare_groups,
    summarise_duplicates,
    summarise_missing,
    summarise_zero_values,
)
from src.metadata import (
    analyse_naming_convention,
    build_dtype_summary,
    describe_file,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
steam_games = load_raw_dataset()

row_count, column_count = steam_games.shape
print(f"Source file : {RAW_DATA_PATH.name}")
print(f"Rows        : {row_count:,}")
print(f"Columns     : {column_count}")

steam_games.head()

Source file : Steam_2024_bestRevenue_1500.csv
Rows        : 1,500
Columns     : 11


,name,releaseDate,copiesSold,price,revenue,avgPlaytime,reviewScore,publisherClass,publishers,developers,steamId
0,WWE 2K24,07-03-2024,165301,99.99,8055097.0,42.365140,71,AAA,2K,Visual Concepts,2315690
1,EARTH DEFENSE FORCE 6,25-07-2024,159806,59.99,7882151.0,29.651061,57,Indie,D3PUBLISHER,SANDLOT,2291060
2,Sins of a Solar Empire II,15-08-2024,214192,49.99,7815247.0,12.452593,88,Indie,Stardock Entertainment,"Ironclad Games Corporation,Stardock Entertainment",1575940
3,Legend of Mortal,14-06-2024,440998,19.99,7756399.0,24.797817,76,Indie,"Paras Games,Obb Studio Inc.",Obb Studio Inc.,1859910
4,Shin Megami Tensei V: Vengeance,13-06-2024,141306,59.99,7629252.0,34.258496,96,AA,SEGA,ATLUS,1875830


In [3]:
verify_selection_requirements(steam_games)

,Requirement,Required,Observed,Status
0,Tabular format,Yes,Yes (single flat CSV table),PASS
1,Minimum rows,>= 1000,1500,PASS
2,Minimum columns,>= 10,11,PASS
3,Numeric variables,>= 1,"5 (copiesSold, price, revenue, avgPlaytime, re...",PASS
4,Temporal variables,>= 1,1 (releaseDate),PASS
5,Categorical variables,>= 1,"3 (publisherClass, publishers, developers)",PASS


In [4]:
build_column_inventory(steam_games)

,Column,Assigned role,Stored dtype,Non-null,Missing,Unique values,Example value
0,name,identifier,str,1500,0,1500,WWE 2K24
1,releaseDate,temporal,str,1500,0,235,07-03-2024
2,copiesSold,numeric,int64,1500,0,1460,165301
3,price,numeric,float64,1500,0,58,99.99
4,revenue,numeric,float64,1500,0,1497,8055097.0
5,avgPlaytime,numeric,float64,1500,0,1500,42.36514
6,reviewScore,numeric,int64,1500,0,72,71
7,publisherClass,categorical,str,1500,0,4,AAA
8,publishers,categorical,str,1499,1,1131,2K
9,developers,categorical,str,1498,2,1406,Visual Concepts


#### What the verification shows

The dataset satisfies every selection criterion, but the two tables above
already raise several points that shape the rest of the analysis.

**The column margin is narrow.** The dataset has 11 columns against a required
minimum of 10. Only one column can be removed before the dataset stops meeting
the specification. This is a real constraint on later sections: a column that
turns out to carry little information cannot simply be dropped, it has to be
retained and its weakness documented instead.

**Two of the eleven columns are identifiers, not variables.** `name` and
`steamId` each hold 1,500 unique values across 1,500 rows. They carry no
distributional signal and will not appear in any statistical summary, but they
define what a row *is*, which makes them central to the duplicate analysis in
section 4 and the index analysis in section 7. Excluding them, only nine columns
are analytically substantive.

**Missing data is almost absent.** Three cells out of 16,500 are empty — around
0.02% — and all three sit in `publishers` and `developers`. A dataset this
complete is itself worth questioning: near-perfect completeness usually
indicates either a curated extract or fields that were imputed upstream rather
than genuinely observed. Section 4 examines which of these applies.

**`avgPlaytime` has 1,500 unique values in 1,500 rows.** A recorded average
playtime across thousands of players would be expected to produce at least some
repeated values, especially at the low end. A column that is unique for every
single row, carried to roughly fourteen decimal places, has the signature of a
*computed* quantity rather than a *recorded* one. This is the first concrete
hint that parts of this dataset are modelled rather than measured — a thread
picked up in section 2.2 and quantified in section 6.

**`releaseDate` is stored as text, not as a date.** The values follow a
`DD-MM-YYYY` layout, which pandas does not recognise automatically. Parsing it
requires an explicit format: relying on inference would silently misread every
date where both day and month are 12 or below, for example `01-02-2024`.
Section 3 performs this conversion deliberately for that reason.

### 2.2 Description of the Data Source

#### Origin of the data

The file was downloaded from **Kaggle** as a static CSV snapshot. No datasheet,
licence file, methodology note or version tag accompanied the download.

> **To be completed before submission:** the exact Kaggle dataset URL and its
> stated licence.

#### The underlying source, and why the numbers cannot be measurements

Valve does not publish per-title sales or revenue figures for Steam. No public
interface exposes how many copies a given game has sold. It follows that the
`revenue` and `copiesSold` columns **cannot be direct measurements**; they must
be estimates produced by a third-party analytics provider.

The established industry approach is to infer the number of owners from the
publicly visible review count using a review-to-owner multiplier — widely known
as the *Boxleiter method* — and then combine that estimate with observed pricing
to obtain revenue. Two pieces of internal evidence are consistent with the file
having been produced this way:

1. **`avgPlaytime` is unique for all 1,500 rows**, carried to roughly fourteen
   decimal places. Recorded averages do not behave like this; derived model
   outputs do.
2. **`revenue` is close to, but not identical to, `copiesSold` × `price`.** If
   revenue were a simple product of the two, the relationship would be exact. It
   is not, and a subset of rows even exceeds that product — which is what one
   would expect from a model that accounts for discount and regional price
   history rather than applying a single current price. Section 6 quantifies
   this relationship.

This distinction matters more than it may appear. It means the dataset does not
record what happened on Steam in 2024; it records **what one estimation model
believes happened**, and every downstream conclusion inherits that model's
assumptions and its unstated error.

#### Purpose of collection

No stated purpose ships with the file, so the purpose must be inferred from its
shape. The selected fields — revenue, copies sold, price, publisher class — are
precisely the fields required for commercial market intelligence, and the
ranking-and-truncation to a "top 1,500" is a market-report convention rather
than a sampling design. A dataset assembled for academic research would more
plausibly retain genre, tags, supported platforms, languages and raw review
counts, and would not discard everything below a revenue threshold.

The evidence therefore points to **commercial rather than scientific**
collection. Section 3 develops this distinction and its consequences in full.

#### The collecting entity

This is only partially answerable. The immediate distributor is the Kaggle
uploader; the upstream producer is an analytics vendor that the file does not
name. Which vendor, and which version of its model, cannot be determined from
the data alone. **The collecting entity is therefore effectively unknown**, and
this is recorded as a limitation rather than presented as resolved.

#### Domain context

Steam is the dominant storefront for PC games, which makes a top-revenue Steam
extract a reasonable proxy for the commercial upper tier of PC gaming — but only
for that tier. Steam receives many thousands of new titles per year, so 1,500
titles represent a small fraction of 2024 releases, and by construction the most
commercially successful fraction.

> **To be completed before submission:** cite a source for the total number of
> Steam releases in 2024, so the coverage fraction can be stated precisely
> rather than qualitatively.

One further ambiguity is material to any financial reading: revenue of this kind
is normally quoted **gross**, before Valve's platform cut, refunds and taxes.
The file does not state whether that is the case here. The difference between
gross and net is on the order of 30%, which is larger than most of the effects
this analysis will detect.

#### What the source does not tell us

The assignment notes that a lack of information is itself an insight. Here the
gaps are substantial, and each one is a question the data cannot answer:

| Unanswered question | Why it matters |
|---|---|
| On what date was the snapshot taken? | Revenue accumulates over time; without a snapshot date, the exposure window of each title is unknown. |
| Is revenue gross or net? | A roughly 30% difference in every monetary figure. |
| Which currency, and how were regions converted? | Affects comparability across titles. |
| Is `price` the launch price, the current price, or an average? | Determines whether price and revenue are even mutually consistent. |
| Is revenue lifetime-to-date or 2024-only? | Changes the meaning of every comparison against release date. |
| What is the estimation method, and what is its error? | No confidence interval can be reconstructed. |
| Under what licence is the data released? | Governs whether the analysis may be published. |

The practical consequence is a boundary on what this notebook may legitimately
claim: the figures support **relative** statements — comparisons, rankings,
associations between variables — but they do not support **absolute** financial
claims about any individual title.

#### The selection principle, and its consequence

The filename encodes the sampling rule: `bestRevenue_1500`. The rows were
selected *because* they earned the most. This is a truncated, ranked, top-of-
distribution sample, and it is the single most important fact about the dataset.

Every distributional statement in this notebook therefore describes **the
top 1,500 earning games of 2024**, never "games on Steam". A statement such as
"the average game earned 2.6 million dollars" would be false by construction:
the sample was selected on the very quantity being averaged. This selection
effect is revisited as a formal bias in section 8.

#### Motivation for choosing this dataset

> *[TO BE PERSONALISED]* — the assignment asks for a topic close to the
> student's own interests. Replace this paragraph with your own reason for
> choosing the Steam games market.

## 3. Meta-Analysis of the Data

Before reading any value in the table, this section examines the dataset as an
**artefact**: a file, produced by some process, at some point in time. Two of
the most consequential findings in this entire notebook come from the file's
own metadata rather than from its contents.

### 3.1 File Analysis

In [5]:
describe_file(RAW_DATA_PATH)

,Property,Value
0,File name,Steam_2024_bestRevenue_1500.csv
1,Format,CSV — delimited plain text
2,Size on disk,"180,690 bytes (176.5 KB)"
3,Encoding,UTF-8 (contains non-ASCII characters)
4,Byte order mark,Absent
5,Line terminator,CRLF (\r\n) — Windows convention
6,Field delimiter,"Comma, with quoting for fields containing commas"
7,Quote characters,734
8,Header row,Present (first line holds column names)
9,Total lines,"1,501 (1 header + 1,500 data rows)"


#### What the file itself reveals

**The creation timestamp recovers the snapshot date.** Section 2.2 listed "on
what date was the snapshot taken?" as a question the data could not answer.
The filesystem answers it: the file was created on **2024-09-11**, and the
latest release date it contains is **2024-09-06** — five days earlier. The two
are consistent with an extract pulled on or about 11 September 2024.

This inference deserves a caveat. A filesystem timestamp records when *this
copy* was written, which is the download, not necessarily the moment the vendor
generated the data. What it gives is an **upper bound** on the age of the
contents, and the narrow five-day gap to the most recent release makes it a
credible proxy for the true extraction date rather than a coincidence.

**The consequence is severe, and it shapes everything that follows.** If
revenue was accumulated up to roughly 11 September 2024, then every title was
observed over a *different* window:

- a game released on 1 January had about **254 days** to earn
- a game released on 6 September had about **5 days**

That is a fiftyfold difference in exposure. `revenue` is therefore not a
property of a game alone; it is a property of a game *and* of how long it had
been on sale when the snapshot was taken. Any comparison of revenue across
release dates conflates commercial success with time on the market. This
confound is revisited in sections 6, 7 and 9.

**The file was produced on Windows.** Every line ends with CRLF. On its own
this is trivia, but combined with the naming conventions examined in 3.2 it
helps place the export pipeline.

**The text is UTF-8 and genuinely international.** Company and title fields
contain trademark symbols, CJK characters and even emoji. Any text handling in
later sections must not assume ASCII, and the presence of non-Latin studio
names matters for section 8: review-based estimation is known to behave
differently across language markets.

**The format carries no schema.** A CSV declares no types, no units, no
currency and no missing-value convention. Every such fact in this notebook is
*inferred* rather than read, which is precisely why section 3.2 examines the
column names so closely — they are the only documentation that exists.

### 3.2 Data Structure

The dataset is 1,500 rows by 11 columns, as established in section 2.1. This
section asks a different question: do the names and types make sense, and what
does the answer reveal about where the data came from?

In [6]:
structure_summary = build_dtype_summary(steam_games).merge(
    analyse_naming_convention(steam_games), on="Column"
)
structure_summary

,Column,Stored dtype,Python type of value,Memory (KB),Naming style,Reads as plural
0,name,str,str,108.6,camelCase,No
1,releaseDate,str,str,86.4,camelCase,No
2,copiesSold,int64,int64,11.7,camelCase,No
3,price,float64,float64,11.7,camelCase,No
4,revenue,float64,float64,11.7,camelCase,No
5,avgPlaytime,float64,float64,11.7,camelCase,No
6,reviewScore,int64,int64,11.7,camelCase,No
7,publisherClass,str,str,78.6,camelCase,No
8,publishers,str,str,96.0,camelCase,Yes
9,developers,str,str,95.7,camelCase,Yes


In [7]:
summarise_entity_columns(steam_games, ["publishers", "developers"])

,Column,Distinct stored strings,Distinct companies,Rows holding >1 company,Most companies in one row
0,publishers,1131,1169,174,3
1,developers,1406,1517,101,6


In [8]:
release_dates = parse_release_date(steam_games)

# The file's modification time is the best available proxy for the date on
# which the data was extracted, as argued in section 3.1.
snapshot_date = pd.Timestamp(RAW_DATA_PATH.stat().st_mtime, unit="s").normalize()
exposure_days = (snapshot_date - release_dates).dt.days

print(f"Dates that failed to parse : {release_dates.isna().sum()}")
print(f"Earliest release           : {release_dates.min():%Y-%m-%d}")
print(f"Latest release             : {release_dates.max():%Y-%m-%d}")
print(f"Calendar months covered    : {sorted(int(m) for m in release_dates.dt.month.unique())}")
print()
print(f"Assumed snapshot date      : {snapshot_date:%Y-%m-%d}")
print(f"Exposure window (days)     : {exposure_days.min()} to {exposure_days.max()}")

Dates that failed to parse : 0
Earliest release           : 2024-01-01
Latest release             : 2024-09-06
Calendar months covered    : [1, 2, 3, 4, 5, 6, 7, 8, 9]

Assumed snapshot date      : 2024-09-11
Exposure window (days)     : 5 to 254


#### What the structure reveals

**The column names are perfectly consistent, and that is evidence.** All eleven
are camelCase, with no exceptions and no mixed styles. Hand-assembled
spreadsheets do not look like this; names generated from a typed schema do.
camelCase is also the dominant convention in JSON, which points to a CSV
serialised from an API response rather than exported from a relational
database, where `snake_case` would be more usual.

**The names look self-explanatory but are not sufficient.** They pass the
"do these make sense?" test easily — and that is exactly the trap. Not one of
them declares a unit:

- `revenue` and `price` state **no currency**. USD is a reasonable guess for a
  Steam extract, but it is a guess.
- `avgPlaytime` states **no unit**. Values average 12.6 with a maximum of 296,
  which is only plausible as hours; in minutes the typical game would be played
  for under thirteen minutes.
- `reviewScore` states **no definition**. It ranges from 0 to 100 and is most
  plausibly the percentage of positive reviews, but it could equally be a
  composite rating.

Readable names created a false sense of documentation. Every unit above is an
inference, and section 4 shows that at least one of these columns uses `0` to
mean two entirely different things.

**`publishers` and `developers` are the only plural names, and the plural is
literal.** These columns are *not* atomic categories. 174 rows list more than
one publisher and 101 list more than one developer, up to six companies in a
single cell. Splitting them properly raises the count of distinct publishers
from 1,131 to 1,169 and distinct developers from 1,406 to 1,517.

Separating them is subtler than splitting on commas, because some legal names
contain their own comma — `CAPCOM Co., Ltd.` is one company, while
`Aspyr,Crystal Dynamics` is two. In this file the two cases are cleanly
separable: a list separator is never followed by a space, while a comma inside
a legal name always is. `split_entity_list` in `src/data_loading.py` encodes
that rule, including for values such as `Cygames, Inc.,Arc System Works` which
contain both kinds at once.

The practical consequence is that any naive frequency count over these columns
is wrong: it would treat `Aspyr,Crystal Dynamics` as a category unrelated to
`Aspyr`. Section 5.2 handles them as multi-valued fields.

**Two dtypes misrepresent what they store.**

- `releaseDate` is text. Parsing it with an explicit `DD-MM-YYYY` format
  succeeds for all 1,500 rows with no failures. Relying on inference would have
  been dangerous rather than merely untidy: pandas would silently read
  `01-02-2024` as 2 January instead of 1 February.
- `steamId` is stored as `int64`, which invites arithmetic that is meaningless —
  the mean of an identifier has no interpretation. It is a label that happens to
  be written with digits, and section 7 treats it accordingly.

**`publisherClass` has only four values and an implicit order.** Stored as free
text, it is really an ordinal scale — Hobbyist, Indie, AA, AAA ascending by
studio scale. Nothing in the file records that ordering, so it has to be
supplied from domain knowledge.

**The dataset costs three times more in memory than on disk** — about 545 KB
against 176 KB — because text columns are stored as Python objects. At this
size it is irrelevant, but it is the mechanism behind memory problems on
datasets a thousand times larger.

#### Discussion: were these data collected for research or for operations?

The evidence points consistently to **commercial rather than scientific**
collection.

1. **The sampling rule is a business rule.** A research dataset would sample
   randomly or aim to be exhaustive. This one keeps the top 1,500 by revenue —
   and 1,500 is a round product number, not a statistical criterion.
2. **The field selection is commercially motivated.** Every retained column is
   directly actionable for a publisher or investor. Absent are the fields a
   researcher would need first: genre, tags, supported platforms, languages,
   release region, DLC and — most tellingly — **review count**.
3. **The most diagnostic omission is review count.** Section 2.2 argued that
   sales estimates of this kind are derived from review counts. The file keeps
   `reviewScore` (a percentage) but discards the review *count* — the very
   quantity the estimate is built from. Whether deliberate or incidental, the
   effect is that the output of the model is published while its main input is
   withheld, which makes the estimate impossible to audit.
4. **No methodology, uncertainty or provenance ships with the data.** Research
   data is expected to carry a datasheet; product feeds are not.
5. **The serialisation looks like an API product feed** — camelCase names, flat
   structure, no schema file.

**The biases that follow from this**, each of which constrains later sections:

| Bias | Mechanism | Where it bites |
|---|---|---|
| **Selection / survivorship** | Rows chosen *because* revenue was high | Every distributional claim (§5, §8) |
| **Estimation** | Values are model output, not measurement, with unknown systematic error | All monetary analysis (§6) |
| **Exposure / right-censoring** | 5 to 254 days of accumulation depending on release date | Any revenue-versus-time comparison (§6, §7) |
| **Visibility** | Review-driven estimates favour games whose players review, which varies by genre, price and language market | Cross-segment comparison (§6.2) |
| **Business-model** | 85 titles priced at 0 still report revenue, up to $102M — their figures cannot come from price × copies and must follow a different estimation path | Price and revenue analysis (§5, §6) |
| **Temporal truncation** | Coverage stops on 6 September; **the entire fourth quarter is missing** | Any seasonal claim (§7, §9) |

The final row deserves emphasis. Q4 contains the Black Friday and winter sales,
commercially the most important weeks of the year for PC games. A dataset of
"2024's top games" that stops in September does not describe 2024; it describes
the first two-thirds of it, and it systematically under-represents titles timed
for the holiday season.

## 4. Data Quality and Completeness

Section 3 established what the dataset *claims* to contain. This section tests
whether it holds up. The central question is not "how many values are missing"
— very few are — but whether the values that are present mean what they appear
to mean.

### 4.1 Missing Data

In [9]:
summarise_missing(steam_games)

,Column,Missing,Share of column,Share of all cells
0,publishers,1,0.067%,0.0061%
1,developers,2,0.133%,0.0121%


In [10]:
incomplete_rows = steam_games[steam_games[["publishers", "developers"]].isna().any(axis=1)]
incomplete_rows[["name", "publisherClass", "publishers", "developers", "revenue", "steamId"]]

,name,publisherClass,publishers,developers,revenue,steamId
643,YUME 4,Indie,Lovely Games,NaN,54457.0,2602730
710,Pixel Noir,Hobbyist,NaN,SWDTech Games,47871.0,754320
765,Hypnosis Card,Indie,Lovely Games,NaN,41891.0,2544990


#### Extent, pattern, and what the absence itself teaches

**The extent is negligible: three cells out of 16,500, or 0.018%.** Only
`publishers` (1) and `developers` (2) are affected; the other nine columns are
complete. On extent alone this would be a two-line finding.

**The pattern, however, is structured rather than random — and with only three
cases the structure is unusually legible.**

*The missing publisher is not an error.* It belongs to **Pixel Noir**, which is
the **only row in the entire dataset classified as `Hobbyist`**. One row of one
kind, and it is exactly the row where a publisher is absent. A hobbyist release
by definition has no publishing company, so the blank does not mean "unknown" —
it means **"none exists"**. The value is correctly absent.

*The two missing developers share a publisher.* Both **YUME 4** and **Hypnosis
Card** are published by **Lovely Games**. Two out of two pointing at a single
company is not plausible chance; it indicates that one upstream record set was
incomplete, not that developer information is randomly lost.

*All three rows are small.* They earn $54k, $48k and $42k, and all are Indie or
Hobbyist. Metadata coverage is worse at the small end of the market — the same
visibility bias identified in section 3, appearing here in a second form.

**This missingness is therefore informative, not merely inconvenient.** In the
standard terminology it is *missing not at random*: whether a value is absent
depends on what the value would have been. The blank in `publishers` carries
real information — that the game is self-published — which a "complete" dataset
using a placeholder string would have hidden.

#### How the gaps should be filled

The answer differs by column, and in neither case is a statistical imputation
appropriate.

For **`publishers`**, nothing should be imputed. The correct treatment is an
explicit category such as `"Self-published"`, which preserves the fact rather
than disguising it as a gap.

For **`developers`**, imputation is tempting: 49% of rows in this dataset list
the same company as publisher and developer, so "Lovely Games" is a defensible
guess. It is still a guess, and it would **attribute two real games to a real
company on the basis of a correlation**. The honest options are to leave the
values missing and exclude the rows from developer-level counts, or to mark
them `"Unknown"`.

The general point matters more than these two rows: mean or mode imputation is
meaningless for identity fields. Filling `developers` with the most common
studio would fabricate an authorship claim. Imputation strategy has to follow
from what a column *is*, not from its dtype.

### 4.2 Duplicates

In [11]:
duplicate_report = summarise_duplicates(
    steam_games,
    {
        "Steam app ID": ["steamId"],
        "Game title": ["name"],
        "Title + developer + release date": ["name", "developers", "releaseDate"],
        "Revenue value": ["revenue"],
    },
)
duplicate_report

,Duplicate check,Columns compared,Repeated rows
0,Entire row (all 11 columns),11,0
1,Steam app ID,1,0
2,Game title,1,0
3,Title + developer + release date,3,0
4,Revenue value,1,3


In [12]:
tied_revenue = steam_games[steam_games["revenue"].duplicated(keep=False)].sort_values("revenue")
tied_revenue[["name", "copiesSold", "price", "revenue", "publisherClass"]]

,name,copiesSold,price,revenue,publisherClass
1066,鬼打墙（剧情版）,3120,5.99,21939.0,Indie
1067,ワールド・ネバーランド２～プルト共和国物語～EXPERIENCE OF FICTION LIFE,1272,24.99,21939.0,Indie
933,Survival Nation: Lost Horizon,2465,14.99,28857.0,Indie
934,Tactic Boxing,3970,9.99,28857.0,Indie
876,飄流幻境M,13101,0.00,32631.0,Indie
877,Horny Suika: Wet Watermelon,7860,4.99,32631.0,Indie


#### Full duplicates, partial duplicates, and one instructive coincidence

**There are no full-row duplicates, and no partial duplicates on any candidate
key.** `steamId` repeats zero times, so it is a genuine primary key. `name`
repeats zero times as well, and the composite of title, developer and release
date is likewise unique. The dataset holds 1,500 distinct games, and no
deduplication is required.

**The only repeated values sit in `revenue`: three values, each shared by
exactly two games.** These are worth examining, because they are not duplicate
records — they are two different games that happen to be assigned the same
figure. The pairs are revealing:

| Revenue | Game A | Game B |
|---|---|---|
| $21,939 | 3,120 copies at $5.99 | 1,272 copies at $24.99 |
| $28,857 | 2,465 copies at $14.99 | 3,970 copies at $9.99 |
| $32,631 | 13,101 copies at **$0.00** | 7,860 copies at $4.99 |

**Two entirely different quantity-and-price combinations produce byte-identical
revenue.** Multiplying copies by price gives $18,689 and $31,787 for the first
pair — neither equals the $21,939 recorded. This is independent confirmation of
the argument in section 2.2: **`revenue` is not computed as `copiesSold` ×
`price`**, but arrives from a separate estimation path. The third pair makes it
starkest, since one of the two games is free.

The ties themselves are unremarkable once that is understood. Revenue is
rounded to whole units and the low-revenue end of this dataset is densely
packed, so occasional collisions are expected rather than suspicious.

**Should the duplicates be removed?** No — and this is the more important
methodological point. Deduplicating on `revenue` would delete three legitimate,
distinct games because they share a rounded figure with another title.
Deduplication must be performed on a key, never on a measured value. Here the
key is `steamId`, and it is already clean.

One structural clue emerges from this check: the three tied pairs sit
**adjacent** to each other in the file, which says something about how the rows
are ordered. Section 7 follows that thread and shows the file is not ordered
the way its name implies.

### 4.3 Suspicious Values

No column contains a negative number, and no value falls outside its plausible
range: `reviewScore` stays within 0–100, `price` within $0–$99.99, and the
minimum `copiesSold` is 593. The problems in this dataset are subtler than
impossible values — they concern **zero**, which appears in three columns and
means something different in each.

In [13]:
summarise_zero_values(steam_games, ["price", "reviewScore", "avgPlaytime", "copiesSold", "revenue"])

,Column,Zeros,Share of rows,Smallest non-zero value
0,price,85,5.7%,1.990000
1,reviewScore,99,6.6%,11.000000
2,avgPlaytime,1,0.1%,0.549644
3,copiesSold,0,0.0%,593.000000
4,revenue,0,0.0%,20674.000000


In [14]:
score_is_zero = steam_games["reviewScore"] == 0

print(f"Games scoring exactly 0            : {score_is_zero.sum()}")
print(f"Games scoring between 1 and 30     : {steam_games['reviewScore'].between(1, 30).sum()}")
print()
print(f"Mean review score including zeros  : {steam_games['reviewScore'].mean():.2f}")
print(f"Mean review score excluding zeros  : {steam_games.loc[~score_is_zero, 'reviewScore'].mean():.2f}")
print()

compare_groups(
    steam_games,
    score_is_zero,
    ["copiesSold", "revenue", "avgPlaytime", "price"],
    ("score is 0", "score above 0"),
)

Games scoring exactly 0            : 99
Games scoring between 1 and 30     : 6

Mean review score including zeros  : 76.20
Mean review score excluding zeros  : 81.59



,Column,Median — score is 0,Median — score above 0
0,copiesSold,30388.00,11023.00
1,revenue,267621.00,101261.00
2,avgPlaytime,8.36,6.69
3,price,16.99,14.99


In [15]:
implied_revenue = steam_games["copiesSold"] * steam_games["price"]
exceeds_implied = steam_games["revenue"] > implied_revenue + 1
is_paid = steam_games["price"] > 0

print(f"Rows where revenue exceeds copiesSold x price : {exceeds_implied.sum()}")
print(f"  of which are free-to-play (price = 0)       : {(exceeds_implied & ~is_paid).sum()}")
print(f"  of which are paid games                     : {(exceeds_implied & is_paid).sum()} of {is_paid.sum()}")
print()

stray_whitespace = steam_games.loc[
    steam_games["name"] != steam_games["name"].str.strip(), "name"
]
print(f"Titles with leading or trailing whitespace    : {len(stray_whitespace)}")
print(stray_whitespace.tolist())

steam_games.loc[
    steam_games["avgPlaytime"] == 0,
    ["name", "copiesSold", "price", "revenue", "reviewScore", "avgPlaytime"],
]

Rows where revenue exceeds copiesSold x price : 92
  of which are free-to-play (price = 0)       : 85
  of which are paid games                     : 7 of 1415

Titles with leading or trailing whitespace    : 2
['Astral Party ', 'SaGa Emerald Beyond ']


,name,copiesSold,price,revenue,reviewScore,avgPlaytime
1186,The Elder Scrolls Online: Gold Road,27601,39.99,960791.0,44,0.0


#### The same zero, three different meanings

**`price` = 0 is a real value (85 rows, 5.7%).** These are free-to-play titles,
and all 85 report revenue above zero — up to $102 million for *The First
Descendant*. The zero is accurate: the game genuinely costs nothing, and the
money arrives through in-game purchases. Two consequences follow. First, `price`
is not a single distribution but a mixture — a spike at zero plus a paid
distribution — so section 5 must treat it conditionally. Second, no model of the
form `revenue = copies x price` can reproduce these rows, which is why 85 of the
92 rows whose revenue exceeds `copiesSold` × `price` are exactly these games.
Among the 1,415 paid games, only **7** break that relationship.

**`reviewScore` = 0 is a placeholder (99 rows, 6.6%), and treating it as a score
is a real error.** Two independent lines of evidence establish this:

*The distribution has a cliff, not a tail.* 99 games sit at exactly 0, while the
entire range from 1 to 30 contains **6 games**. A genuine 0% rating would be the
bottom of a continuum, with neighbours just above it. Instead there is a spike
at zero and near-emptiness beside it — the signature of a reserved value.

*The group behaves in a way its stated meaning cannot explain.* Games "rated 0%"
outsell the rest of the dataset:

| Measure | Score = 0 | Score > 0 |
|---|---|---|
| Median copies sold | 30,388 | 11,023 |
| Median revenue | $267,621 | $101,261 |

Games supposedly rated 0% positive sell **2.8 times more copies** and earn
**2.6 times more** than games with a real score. Universally condemned games do
not outperform the market. The only coherent reading is that 0 encodes **"no
score available"** — most plausibly too few reviews to compute one — and that it
has been written into the same field as genuine scores.

The cost of ignoring this is measurable: the mean review score is **76.2** with
the placeholders included and **81.6** once they are excluded. Leaving them in
biases the mean down by more than five points, and would distort every
correlation involving `reviewScore` in section 6. These 99 values must be
converted to `NaN` before the column is used.

**`avgPlaytime` = 0 is a placeholder too, in a single row.** It belongs to
*The Elder Scrolls Online: Gold Road* — 27,601 copies, $960,791 in revenue and a
review score of 44. An MMO expansion cannot have an average playtime of exactly
zero across 27,601 buyers. The likely mechanism is that playtime accrues to the
base game rather than to the expansion, so the field had nothing to report.

**Two titles carry trailing whitespace** — `'Astral Party '` and
`'SaGa Emerald Beyond '`. Harmless as displayed, but if a title were ever used
as a join key these would silently fail to match their clean counterparts.

**The wider lesson.** Every one of these columns is numeric, complete and free
of nulls. A completeness check alone gives this dataset a clean bill of health,
because the flaws were encoded as *valid numbers*. Missingness disguised as data
is more dangerous than missingness left visible: `NaN` propagates and announces
itself, whereas a placeholder `0` silently enters every mean, correlation and
plot.

### 4.4 Cardinality

In [16]:
build_cardinality_summary(steam_games)

,Column,Distinct values,Uniqueness,Constant,Most frequent value,Its share of rows
0,name,1500,100.0%,No,WWE 2K24,0.1%
1,releaseDate,235,15.7%,No,07-03-2024,1.8%
2,copiesSold,1460,97.3%,No,3127,0.2%
3,price,58,3.9%,No,19.99,15.3%
4,revenue,1497,99.8%,No,32631.0,0.1%
5,avgPlaytime,1500,100.0%,No,42.36514,0.1%
6,reviewScore,72,4.8%,No,0,6.6%
7,publisherClass,4,0.3%,No,Indie,86.7%
8,publishers,1131,75.4%,No,Kagura Games,1.1%
9,developers,1406,93.7%,No,Lust Desires 🖤,0.6%


#### Columns without variance, and columns with too much

**No column is constant**, so nothing can be dropped for lack of variance.
`publisherClass` comes closest with four distinct values, but it is not
degenerate in the literal sense.

**It is, however, close to degenerate in the practical sense.** `Indie` covers
**86.7%** of the rows, and `Hobbyist` covers a single row. A four-level factor
in which one level holds seven rows in eight and another holds one row in 1,500
carries far less information than its four categories suggest. Section 5.2
examines whether the rare levels should be merged.

**Three columns are 100% unique**, and they are unique for two different
reasons. `name` and `steamId` are identifiers, so uniqueness is exactly what
they are for. `avgPlaytime` is not an identifier, and its perfect uniqueness
across 1,500 rows — first noted in section 2.1 — remains the strongest internal
evidence that the column is computed rather than observed.

**`publishers` and `developers` are high-cardinality categoricals**: 1,131 and
1,406 distinct stored strings, rising to 1,169 and 1,517 actual companies once
the delimited lists from section 3.2 are expanded. With roughly as many
categories as rows, these columns cannot be one-hot encoded and will not support
per-category statistics. Any use of them requires aggregation first — by
publisher class, by whether the game is self-published, or by counting a
studio's releases.

**`price` is the opposite case**: only 58 distinct values across 1,500 games,
with `$19.99` alone accounting for **15.3%** of rows. Prices cluster on
psychological price points rather than varying continuously. Section 5 has to
treat `price` as effectively discrete, and correlation measures that assume
continuity will be affected by the resulting ties.

**`releaseDate` covers 235 distinct days out of 250 calendar days** — releases
occur on almost every day of the period, with the busiest being 7 March 2024 at
27 games. Section 7 returns to this.

## 5. Univariate Analysis

This is the first section that consumes the findings of section 4 rather than
merely reporting them. Before any statistic is computed, the two placeholder
zeros identified in 4.3 are converted to missing values, the release date is
parsed, and `publisherClass` becomes an ordered category.

No row is removed. The free-to-play zeros in `price` are also left untouched,
because those zeros are real prices.

In [17]:
games = apply_quality_fixes(steam_games)

summarise_cleaning_effect(steam_games, games, ["reviewScore", "avgPlaytime"])

,Column,Values before,Values after,Marked unknown,Mean before,Mean after
0,reviewScore,1500,1401,99,76.20,81.59
1,avgPlaytime,1500,1499,1,12.56,12.57


### 5.1 Numeric Variables

Each variable is described with both a classical and a robust measure — mean
beside median, standard deviation beside MAD — so that the distance between
them can be read directly from the table. That distance is the finding.

In [18]:
numeric_summary = describe_numeric(games, NUMERIC_COLUMNS)

with pd.option_context("display.float_format", "{:,.2f}".format):
    display(numeric_summary)

,Count,Mean,Median,Std,MAD,Min,Q1,Q3,Max,IQR,Skew
Column,,,,,,,,,,,
copiesSold,1500,"141,482.57","11,928.50","1,132,756.66","8,931.50",593.00,"4,918.75","37,869.75","30,739,148.00","32,951.00",18.62
price,1500,17.52,14.99,12.65,5.00,0.00,9.99,19.99,99.99,10.00,1.58
revenue,1500,"2,632,381.98","109,053.00","27,810,239.62","80,258.50","20,674.00","45,504.25","455,156.75","837,793,356.00","409,652.50",22.92
avgPlaytime,1499,12.57,6.76,21.55,3.99,0.55,3.57,13.11,296.33,9.54,7.17
reviewScore,1401,81.59,85.00,13.92,8.00,11.00,75.00,92.00,100.00,17.00,-1.31


#### Location, spread and skew

**Four of the five variables are strongly right-skewed, and two of them
extremely so.**

| Variable | Mean | Median | Mean ÷ median | Std ÷ MAD | Skew |
|---|---|---|---|---|---|
| `revenue` | $2,632,382 | $109,053 | **24×** | **347×** | **22.9** |
| `copiesSold` | 141,483 | 11,929 | **12×** | 127× | **18.6** |
| `avgPlaytime` | 12.57 h | 6.76 h | 1.9× | 5.4× | 7.2 |
| `price` | $17.52 | $14.99 | 1.2× | 2.5× | 1.6 |
| `reviewScore` | 81.59 | 85.00 | 0.96× | 1.7× | **−1.3** |

For `revenue`, the standard deviation is **347 times** the MAD. The two
statistics are meant to measure the same thing — typical distance from the
centre — and they disagree by more than two orders of magnitude. That gap is
not noise; it is the signature of a distribution whose spread is dominated by a
handful of values.

**`reviewScore` is the only left-skewed variable**, and its skew is mild
(−1.31). Scores pile up near the top: the median is 85 and the third quartile
is 92. This is what a top-revenue sample should look like — commercially
successful games are generally well received — combined with a scale that is
bounded at 100 and cannot produce a right tail.

**`price` is the least skewed** because it is doubly constrained: bounded above
by what a PC game can charge, and clustered on a small number of psychological
price points, as section 4.4 showed.

In [19]:
compare_outlier_methods(games, NUMERIC_COLUMNS)

,Values,IQR (1.5x),Z-score (|z|>3),Modified Z (|M|>3.5),IQR (1.5x) %,Z-score (|z|>3) %,Modified Z (|M|>3.5) %
Column,,,,,,,
copiesSold,1500,201,10,274,13.4,0.7,18.3
price,1500,140,34,65,9.3,2.3,4.3
revenue,1500,223,8,346,14.9,0.5,23.1
avgPlaytime,1499,147,24,147,9.8,1.6,9.8
reviewScore,1401,48,17,25,3.4,1.2,1.8


#### Outlier detection: three methods that do not agree

The three methods are applied on the raw scale above, and they disagree
violently. For `revenue` the z-score flags **8** values while the modified
z-score flags **346** — a factor of 43 between two methods that are supposed to
answer the same question.

The disagreement is not a tie to be broken. It is diagnostic, and the reason is
mechanical:

- The **z-score** measures distance from the *mean* in units of the *standard
  deviation*. Both are computed from data that includes the extreme values, so
  the extremes inflate the very yardstick used to judge them. On `revenue`,
  `mean + 3 × std` comes to **$86,063,101 — roughly 789 times the median**.
  Only eight games clear a bar set that high. This is the classic *masking*
  effect: the outliers hide themselves by making the scale enormous.
- The **modified z-score** is built on the median and the MAD, neither of which
  the extremes can inflate, so it is not fooled. But it then flags 23% of the
  dataset, and a category containing a quarter of all observations is not
  usefully described as "outliers".
- The **IQR rule** lands between the two at 14.9%.

So one method is blind, one is overwhelmed, and the third is somewhere in
between. The natural conclusion would be that this dataset is riddled with
outliers. The next cell shows that conclusion is wrong.

In [20]:
# revenue and copiesSold span more than four orders of magnitude, which suggests
# the underlying process is multiplicative rather than additive. If so, the
# analysis belongs on a logarithmic scale.
logged_games = games.assign(
    log_revenue=np.log10(games["revenue"]),
    log_copiesSold=np.log10(games["copiesSold"]),
)

with pd.option_context("display.float_format", "{:,.3f}".format):
    display(describe_numeric(logged_games, ["log_revenue", "log_copiesSold"])[
        ["Mean", "Median", "Std", "MAD", "Skew"]
    ])

compare_outlier_methods(logged_games, ["log_revenue", "log_copiesSold"])

,Mean,Median,Std,MAD,Skew
Column,,,,,
log_revenue,5.235,5.038,0.747,0.452,1.187
log_copiesSold,4.189,4.077,0.698,0.429,0.982


,Values,IQR (1.5x),Z-score (|z|>3),Modified Z (|M|>3.5),IQR (1.5x) %,Z-score (|z|>3) %,Modified Z (|M|>3.5) %
Column,,,,,,,
log_revenue,1500,30,20,21,2.0,1.3,1.4
log_copiesSold,1500,33,17,17,2.2,1.1,1.1


#### The scale was wrong, not the data

Taking base-10 logarithms changes the picture completely:

| | Skew (raw) | Skew (log) | IQR flags (raw) | IQR flags (log) |
|---|---|---|---|---|
| `revenue` | 22.92 | **1.19** | 223 (14.9%) | **30 (2.0%)** |
| `copiesSold` | 18.62 | **0.98** | 201 (13.4%) | **33 (2.2%)** |

Two things happen at once. The distributions become close to symmetric, and —
more tellingly — **the three detection methods stop disagreeing**: on
`log_revenue` they flag 30, 20 and 21 values respectively, all around 1–2%.

That convergence is the answer. The methods were never really in conflict about
the data; they were reacting differently to a mismatch between the tool and the
scale. `revenue` ranges from $20,674 to $837,793,356 — over four orders of
magnitude — because game revenue is generated by a multiplicative process, not
an additive one. Statistics that assume additive structure will misbehave on it
no matter which threshold is chosen.

**This reframes the question the assignment asks.** On the raw scale, "are there
outliers?" is close to ill-posed. The extreme values are not contaminated
measurements to be cleaned away; they are the phenomenon. *Black Myth: Wukong*
is not a data error — it is the single most important observation in the
dataset. Removing the top 2% would delete most of the market being studied.

The defensible position is therefore: **treat these variables on a log scale,
report the ~2% of genuine standouts that survive that transformation, and
retain them.**

### 5.2 Categorical Variables

In [21]:
summarise_categorical(games["publisherClass"])

,Value,Count,Share,Cumulative share
0,Indie,1301,86.73,86.73
1,AA,146,9.73,96.46
2,AAA,52,3.47,99.93
3,Hobbyist,1,0.07,100.00


In [22]:
publisher_mentions = expand_entity_column(games, "publishers")
developer_mentions = expand_entity_column(games, "developers")

print(f"Distinct publishers : {publisher_mentions.nunique():,} across {len(publisher_mentions):,} mentions")
print(f"Distinct developers : {developer_mentions.nunique():,} across {len(developer_mentions):,} mentions")
print(f"Publishers with exactly one game: {(publisher_mentions.value_counts() == 1).sum():,} "
      f"({(publisher_mentions.value_counts() == 1).mean():.1%})")
print()

summarise_categorical(publisher_mentions, top_n=8)

Distinct publishers : 1,169 across 1,683 mentions
Distinct developers : 1,517 across 1,627 mentions
Publishers with exactly one game: 971 (83.1%)



,Value,Count,Share,Cumulative share
0,PlayWay S.A.,22,1.31,1.31
1,Kagura Games,17,1.01,2.32
2,Electronic Arts,16,0.95,3.27
3,Mango Party,14,0.83,4.10
4,072 Project,14,0.83,4.93
5,Gamera Games,13,0.77,5.70
6,Mango Party News,13,0.77,6.47
7,Ubisoft,13,0.77,7.24


In [23]:
print("Publishers")
display(values_needed_for_coverage(publisher_mentions))
print("Developers")
display(values_needed_for_coverage(developer_mentions))

Publishers


,Coverage target,Categories needed,Share of all categories
0,25%,67,5.7%
1,50%,328,28.1%
2,80%,833,71.3%
3,90%,1001,85.6%


Developers


,Coverage target,Categories needed,Share of all categories
0,25%,297,19.6%
1,50%,704,46.4%
2,80%,1192,78.6%
3,90%,1355,89.3%


#### Frequencies, coverage and rare categories

**`publisherClass` has a dominant mode and almost nothing else.** `Indie`
covers **86.7%** of rows, `AA` 9.7%, `AAA` 3.5%, and `Hobbyist` a single game.
The mode is genuinely representative here, but only because the variable is so
lopsided that it carries little information — knowing a game is Indie tells you
what you already knew about seven rows in eight.

**The publisher distribution is remarkably flat, which is the opposite of what
one would expect.** The largest publisher in the dataset, PlayWay S.A., accounts
for **1.31%** of games. The coverage table makes the point precisely:

| To cover this share of games | Publishers needed | Developers needed |
|---|---|---|
| 25% | 67 (5.7% of all publishers) | 297 (19.6%) |
| 50% | **328 (28.1%)** | 704 (46.4%) |
| 80% | 833 (71.3%) | 1,192 (78.6%) |
| 90% | 1,001 (85.6%) | 1,355 (89.3%) |

It takes **328 different companies** to account for half the games, and
**83.1% of publishers appear exactly once**. There is no concentration of
authorship at all — this is a market of many small independent studios.

Hold that beside the revenue figures in the next cell, because the contrast is
the most interesting result in this section.

**Rare categories: the `Hobbyist` problem.** One row in 1,500. Is it
informative or can it be ignored? Both, depending on the use:

- *As a label it is informative.* Section 4.1 showed that this single row is
  precisely the one with a missing publisher, and the category explained why.
  Discarding it would have destroyed that finding.
- *As a statistical category it is unusable.* No variance can be estimated from
  one observation, and any group statistic for `Hobbyist` is simply that game.

The reasonable treatment is to **merge `Hobbyist` into `Indie`** for grouped
analysis — the two describe the same thing at different scales, and the boundary
between them is a vendor's judgement rather than a fact — while keeping the
original column so the 4.1 finding stays recoverable.

**`publishers` and `developers` cannot be used as categories at all.** With
83% singletons and roughly as many categories as rows, no per-category statistic
is meaningful. They must be aggregated first — by class, by whether the studio
self-publishes, or by counting releases per studio.

**One caution on these counts.** Company names are not normalised in the source.
`Mango Party` (14 games) and `Mango Party News` (13 games) are almost certainly
related entities recorded under different strings. The publisher and developer
counts above should therefore be read as close approximations, not exact
company-level truth.

In [24]:
revenue_by_size = games["revenue"].sort_values(ascending=False)
total_revenue = revenue_by_size.sum()

print("Share of all revenue in the dataset held by the highest earners")
for game_count in [1, 5, 10, 50, 150]:
    share_of_revenue = revenue_by_size.head(game_count).sum() / total_revenue
    print(f"  top {game_count:>3} games ({game_count / len(games):>5.1%} of rows): {share_of_revenue:>6.1%}")

print()
print(f"Games earning less than the mean : {(games['revenue'] < games['revenue'].mean()).mean():.1%}")
print(f"Arithmetic mean revenue          : ${games['revenue'].mean():>12,.0f}")
print(f"Geometric mean revenue           : ${10 ** np.log10(games['revenue']).mean():>12,.0f}")
print(f"Median revenue                   : ${games['revenue'].median():>12,.0f}")

Share of all revenue in the dataset held by the highest earners
  top   1 games ( 0.1% of rows):  21.2%
  top   5 games ( 0.3% of rows):  50.5%
  top  10 games ( 0.7% of rows):  61.6%
  top  50 games ( 3.3% of rows):  82.0%
  top 150 games (10.0% of rows):  91.4%

Games earning less than the mean : 92.2%
Arithmetic mean revenue          : $   2,632,382
Geometric mean revenue           : $     171,715
Median revenue                   : $     109,053


#### Discussion: do the central measures represent the data well?

**For `revenue` and `copiesSold`, no — and not by a small margin.**

**92.2% of games earn less than the mean.** A statistic that all but 8% of the
data falls below is not a description of a typical game; it is an artefact of a
few enormous values. The arithmetic mean of $2,632,382 corresponds to no real
game in the dataset — the games nearest to it sit around the 93rd percentile.

The concentration behind that is extreme:

- **One game — *Black Myth: Wukong* — accounts for 21.2% of all revenue** in
  the dataset.
- **Five games account for half of it.**
- The top 50 games, 3.3% of rows, hold **82%**.

And this concentration exists *inside a sample that is already restricted to the
1,500 highest earners*. Across all of Steam it would be far more extreme.

**Which measure should be used instead depends on the question.** The median
($109,053) describes a typical game well. The geometric mean ($171,715) — the
mean computed in log space — is a defensible single-number summary because it
respects the multiplicative structure of the variable. Neither, on its own,
conveys that five titles hold half the money, so a concentration statistic has
to be reported alongside them.

**The other three variables behave better.** `reviewScore` has 38% of values
below its mean, which is close enough to half for the mean to be meaningful —
but only *after* the section 4 correction. With the 99 placeholder zeros left
in, the mean read 76.2 instead of 81.6, and would have misrepresented the
variable by more than five points. `price` has a representative median of
$14.99, though its mean of $17.52 is not a price any game actually charges,
since the top five price points alone cover 48.3% of games.

**The general principle.** A measure of central tendency is only meaningful when
the distribution has a centre. For a multiplicative, heavy-tailed quantity like
revenue there is no typical scale in linear terms, so the mean is not merely
imprecise — it is measuring something that does not exist.

**And the deepest point of the section is the contrast between two of its own
results.** Revenue is extraordinarily concentrated: five games hold half the
money. Authorship is not concentrated at all: it takes 328 companies to account
for half the games, and 83% of publishers appear exactly once. The money obeys a
power law while the people do not. Any statement of the form "the games industry
is dominated by a few big players" is therefore true of revenue and false of
participation — a distinction that a single average would have concealed
completely.

## 6. Correlations and Relationships

*Not yet written.* This section will cover:

- **6.1 Numeric to numeric** — Pearson, Spearman and Kendall correlations with an explanation of why they differ, a correlation matrix and scatter plots.
- **6.2 Categorical to categorical** — contingency tables and Cramér's V, plus categorical-to-numeric relationships via binning.
- **6.3 Graphs** — scatterplot, histogram, bar chart, box plot, violin plot, pie chart, pairplot and heatmap, each with titles, axis labels, a legend and written insights.

## 7. Index Analysis

*Not yet written.* This section will cover:

- Is the index unique? Is it time-based? Does the analysis above change over time? Is the data sorted?
- **Carried forward from section 4.2:** the rows are *not* ordered by revenue as the file name implies. The file is a concatenation of four separately sorted blocks, and the highest-earning game is not the first row.

## 8. Insights and the Data Story

*Not yet written.* This section will cover:

- At least three central insights, at least one bias or risk, possible failure points for engineering and statistical use, and what the analysis changed about my understanding of the domain.

## 9. Extensions

*Not yet written.* This section will cover:

- Time dependence and links to external knowledge about the period, feature engineering, hypothesis testing, and suggestions for further research.